In [2]:
import numpy as np
import os
import cv2
import joblib
import warnings

from sklearn.svm import SVC
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import KernelPCA
from sklearn.metrics import accuracy_score

warnings.filterwarnings("ignore")

dataset_path = "dataset"

X = []
y = []

label_map = {}
label_id = 0


# =============================
# LBP FEATURE
# =============================

def lbp_feature(img):

    h, w = img.shape
    lbp = np.zeros((h-2, w-2), dtype=np.uint8)

    for i in range(1, h-1):
        for j in range(1, w-1):

            center = img[i, j]

            binary = [
                img[i-1,j-1] > center,
                img[i-1,j] > center,
                img[i-1,j+1] > center,
                img[i,j+1] > center,
                img[i+1,j+1] > center,
                img[i+1,j] > center,
                img[i+1,j-1] > center,
                img[i,j-1] > center
            ]

            value = sum([b << k for k, b in enumerate(binary)])

            lbp[i-1,j-1] = value

    hist,_ = np.histogram(lbp.ravel(),256,[0,256])

    return hist


# =============================
# GABOR FEATURE
# =============================

def gabor_feature(img):

    kernels = []

    for theta in np.arange(0, np.pi, np.pi/4):

        kernel = cv2.getGaborKernel(
            (9,9),
            1.0,
            theta,
            np.pi/2,
            0.5,
            0,
            ktype=cv2.CV_32F
        )

        kernels.append(kernel)

    features = []

    for kernel in kernels:

        filtered = cv2.filter2D(img, cv2.CV_32F, kernel)

        features.append(filtered.flatten())

    return np.hstack(features)


# =============================
# LOAD DATASET
# =============================

for person in os.listdir(dataset_path):

    person_path = os.path.join(dataset_path, person)

    if not os.path.isdir(person_path):
        continue

    label_map[label_id] = person

    for img_name in os.listdir(person_path):

        if not img_name.lower().endswith((".jpg",".png",".jpeg")):
            continue

        img_path = os.path.join(person_path, img_name)

        img = cv2.imread(img_path,0)

        if img is None:
            continue

        img = cv2.resize(img,(80,70))
        img = cv2.equalizeHist(img)

        gabor = gabor_feature(img)
        lbp = lbp_feature(img)

        feature = np.hstack([gabor, lbp])

        X.append(feature)
        y.append(label_id)

    label_id += 1


X = np.array(X)
y = np.array(y)

print("Dataset shape:", X.shape)
print("Classes:", label_map)


# =============================
# CONFIG
# =============================

n_runs = 3

best_global_accuracy = 0
best_model = None
best_scaler = None
best_kpca = None


# =============================
# TRAIN LOOP
# =============================

for run in range(n_runs):

    print("\n=========================")
    print("RUN", run+1)
    print("=========================")

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=run,
        stratify=y
    )


    # =============================
    # SCALER
    # =============================

    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)


    # =============================
    # KPCA
    # =============================

    kpca = KernelPCA(
        n_components=80,
        kernel="rbf",
        gamma=0.0005
    )

    X_train_kpca = kpca.fit_transform(X_train_scaled)
    X_test_kpca = kpca.transform(X_test_scaled)


    # =============================
    # GRID SEARCH
    # =============================

    param_grid = {
        "C":[1,10,100],
        "gamma":[1e-3,1e-4,1e-5]
    }

    svm = SVC(kernel="rbf")

    grid = GridSearchCV(
        svm,
        param_grid,
        cv=5,
        n_jobs=-1
    )

    grid.fit(X_train_kpca, y_train)

    model = grid.best_estimator_

    print("Best params:", grid.best_params_)


    # =============================
    # TEST
    # =============================

    y_pred = model.predict(X_test_kpca)

    acc = accuracy_score(y_test, y_pred)

    print("Accuracy:", acc)


    # =============================
    # SAVE BEST
    # =============================

    if acc > best_global_accuracy:

        best_global_accuracy = acc
        best_model = model
        best_scaler = scaler
        best_kpca = kpca


# =============================
# SAVE MODEL
# =============================

print("\nSaving best model...")

os.makedirs("models", exist_ok=True)

joblib.dump(best_model,"models/best_svm_model.pkl")
joblib.dump(best_scaler,"models/scaler.pkl")
joblib.dump(best_kpca,"models/kpca.pkl")
joblib.dump(label_map,"models/label_map.pkl")

print("Best Accuracy:", best_global_accuracy)

print("Model saved!")

Dataset shape: (72, 22656)
Classes: {0: 'dung', 1: 'huy', 2: 'me'}

RUN 1
Best params: {'C': 1, 'gamma': 0.001}
Accuracy: 0.9333333333333333

RUN 2
Best params: {'C': 1, 'gamma': 0.001}
Accuracy: 0.6666666666666666

RUN 3
Best params: {'C': 1, 'gamma': 0.001}
Accuracy: 0.6666666666666666

Saving best model...
Best Accuracy: 0.9333333333333333
Model saved!


1/ Gabor Filter Bank (5×8 = 40 filters)

In [ ]:
# import os
# import cv2
# import numpy as np

# def preprocess(img):
#     img = cv2.resize(img, (80, 70))
#     return img

# def build_gabor_bank():
#     filters = []
    
#     scales = []
#     for m in range(1, 6):
#         freq = (np.pi / 2) * (np.sqrt(2) ** (-(m - 1)))
#         sigma = np.pi / freq
#         scales.append(sigma)
    
#     orientations = np.arange(0, np.pi, np.pi/8)
    
#     for sigma in scales:
#         for theta in orientations:
#             lambd = 10.0 
            
#             kernel = cv2.getGaborKernel(
#                 (21, 21), 
#                 sigma=sigma,
#                 theta=theta,
#                 lambd=lambd,
#                 gamma=0.5,
#                 psi=0
#             )
#             filters.append(kernel)
    
#     return filters

# def apply_gabor(img, filters):
#     feature_maps = []
    
#     for kernel in filters:
#         filtered = cv2.filter2D(img, cv2.CV_32F, kernel)
#         feature_maps.append(filtered)
    
#     return feature_maps

2/ Downsample 4×5 (block average)

In [ ]:
# def downsample_block(img, block_h=5, block_w=4):
#     h, w = img.shape
#     new_h = h // block_h 
#     new_w = w // block_w 
    
#     img = img[:new_h * block_h, :new_w * block_w]
    
#     down = img.reshape(new_h, block_h, new_w, block_w).mean(axis=(1, 3))
    
#     return down

# def extract_feature(img, filters):
#     img = preprocess(img)
#     maps = apply_gabor(img, filters)
    
#     features = []
    
#     for m in maps:
#         down = downsample_block(m, 5, 4)  # block_h=5, block_w=4
#         features.append(down.flatten())
    
#     return np.hstack(features)

3/ ĐỌC DATA

In [ ]:
# dataset_path = r"C:\Users\Lenovo\Desktop\New folder\dataset"
# filters = build_gabor_bank()

# X = []
# y = []

# people = sorted(
#     [p for p in os.listdir(dataset_path)
#      if os.path.isdir(os.path.join(dataset_path, p))]
# )

# for label, person in enumerate(people):
    
#     folder = os.path.join(dataset_path, person)
    
#     for file in os.listdir(folder):
        
#         img_path = os.path.join(folder, file)
#         img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        
#         if img is not None:
#             feat = extract_feature(img, filters)
#             X.append(feat)
#             y.append(label)

# X = np.array(X)
# y = np.array(y)

# print(f"X shape: {X.shape}")  # (400, 11200)

X shape: (72, 11200)


In [ ]:
#  # Train - Test Split

# from sklearn.model_selection import train_test_split

# X_train, X_test, y_train, y_test = train_test_split(
#     X,
#     y,
#     test_size=0.5, 
#     random_state=42,
#     stratify=y
# )

# print("Train shape:", X_train.shape)  # (200, 180)
# print("Test shape :", X_test.shape)   # (200, 180)

Train shape: (36, 11200)
Test shape : (36, 11200)


4/ Whitening

In [ ]:
# from sklearn.preprocessing import StandardScaler

# scaler = StandardScaler()

# X_train_scaled = scaler.fit_transform(X_train)
# X_test_scaled = scaler.transform(X_test)

5/ KPCA

In [ ]:
# from sklearn.decomposition import KernelPCA

# kpca = KernelPCA(
#     n_components=180,
#     kernel='rbf',
#     gamma=0.0001
# )

# X_train = kpca.fit_transform(X_train_scaled)
# X_test = kpca.transform(X_test_scaled) # (400, 180)

# print(X_train)

[[-1.73340127e-01 -2.99319271e-02 -2.51762732e-03 ...  3.78324953e-03
  -4.72711079e-04  0.00000000e+00]
 [-4.56942134e-02  1.01968246e-02  1.14336126e-01 ...  1.20072078e-03
   2.88796704e-03  0.00000000e+00]
 [-1.03883676e-01  9.24111810e-03  1.93478778e-01 ...  9.54268209e-04
   5.89081494e-04  0.00000000e+00]
 ...
 [-2.65980482e-01  6.15464523e-02 -2.40843728e-01 ... -4.06893343e-03
   1.31869805e-03  0.00000000e+00]
 [-1.94877625e-01  1.12651035e-01 -1.20016143e-01 ...  7.58482958e-04
  -8.49492382e-04  0.00000000e+00]
 [ 2.06248239e-01  6.78432524e-01  1.06424838e-01 ... -2.95663565e-01
  -4.72016446e-03  0.00000000e+00]]


6/ MUlti-svm

In [ ]:
# import numpy as np
# import pandas as pd
# import os
# import joblib
# import warnings

# from sklearn.svm import SVC, LinearSVC
# from sklearn.model_selection import train_test_split, GridSearchCV
# from sklearn.preprocessing import StandardScaler
# from sklearn.decomposition import KernelPCA
# from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# warnings.filterwarnings("ignore")

# # =============================
# # CONFIG
# # =============================

# n_runs = 3
# all_results = []

# os.makedirs("models", exist_ok=True)

# best_global_accuracy = 0
# best_global_model = None
# best_scaler = None
# best_kpca = None
# best_method = None
# best_kernel = None


# def update_best_model(accuracy, model, scaler, kpca, method, kernel):

#     global best_global_accuracy
#     global best_global_model
#     global best_scaler
#     global best_kpca
#     global best_method
#     global best_kernel

#     if accuracy > best_global_accuracy:

#         best_global_accuracy = accuracy
#         best_global_model = model
#         best_scaler = scaler
#         best_kpca = kpca
#         best_method = method
#         best_kernel = kernel


# # =============================
# # TRAIN LOOP
# # =============================

# for run in range(n_runs):

#     print(f"\n{'='*50}")
#     print(f"RUN {run+1}/{n_runs}")
#     print(f"{'='*50}")

#     X_train, X_test, y_train, y_test = train_test_split(
#         X,
#         y,
#         test_size=0.5,
#         random_state=run,
#         stratify=y
#     )

#     # =============================
#     # Scaling
#     # =============================

#     scaler = StandardScaler()

#     X_train_scaled = scaler.fit_transform(X_train)
#     X_test_scaled = scaler.transform(X_test)

#     # =============================
#     # KPCA
#     # =============================

#     kpca = KernelPCA(
#         n_components=180,
#         kernel="rbf",
#         gamma=0.0001
#     )

#     X_train_kpca = kpca.fit_transform(X_train_scaled)
#     X_test_kpca = kpca.transform(X_test_scaled)

#     # =============================
#     # OAA
#     # =============================

#     for kernel in ["rbf", "poly", "linear"]:

#         print(f"\n▶ OAA + {kernel}")

#         if kernel == "rbf":

#             param_grid = {
#                 "C":[1,10,100],
#                 "gamma":[1e-3,1e-4,1e-5]
#             }

#         elif kernel == "poly":

#             param_grid = {
#                 "C":[1,10,100],
#                 "degree":[2,3,4]
#             }

#         else:

#             param_grid = {
#                 "C":[1,10,100]
#             }

#         svm = SVC(kernel=kernel, decision_function_shape="ovr")

#         grid = GridSearchCV(
#             svm,
#             param_grid,
#             cv=5,
#             n_jobs=-1,
#             scoring="accuracy"
#         )

#         grid.fit(X_train_kpca, y_train)

#         best_model = grid.best_estimator_

#         y_pred = best_model.predict(X_test_kpca)

#         accuracy = accuracy_score(y_test, y_pred)

#         precision_macro = precision_score(y_test, y_pred, average="macro", zero_division=0)
#         recall_macro = recall_score(y_test, y_pred, average="macro", zero_division=0)
#         f1_macro = f1_score(y_test, y_pred, average="macro", zero_division=0)

#         precision_weighted = precision_score(y_test, y_pred, average="weighted", zero_division=0)
#         recall_weighted = recall_score(y_test, y_pred, average="weighted", zero_division=0)
#         f1_weighted = f1_score(y_test, y_pred, average="weighted", zero_division=0)

#         update_best_model(
#             accuracy,
#             best_model,
#             scaler,
#             kpca,
#             "OAA",
#             kernel
#         )

#         all_results.append([
#             run+1,"OAA",kernel,
#             grid.best_score_,
#             accuracy,
#             precision_macro,
#             recall_macro,
#             f1_macro,
#             precision_weighted,
#             recall_weighted,
#             f1_weighted
#         ])

#         print(f"Accuracy: {accuracy:.4f}")


#     # =============================
#     # OAO
#     # =============================

#     for kernel in ["rbf","poly","linear"]:

#         print(f"\n▶ OAO + {kernel}")

#         if kernel == "rbf":

#             param_grid = {
#                 "C":[1,10,100],
#                 "gamma":[1e-3,1e-4,1e-5]
#             }

#         elif kernel == "poly":

#             param_grid = {
#                 "C":[1,10,100],
#                 "degree":[2,3,4]
#             }

#         else:

#             param_grid = {
#                 "C":[1,10,100]
#             }

#         svm = SVC(kernel=kernel, decision_function_shape="ovo")

#         grid = GridSearchCV(
#             svm,
#             param_grid,
#             cv=5,
#             n_jobs=-1,
#             scoring="accuracy"
#         )

#         grid.fit(X_train_kpca, y_train)

#         best_model = grid.best_estimator_

#         y_pred = best_model.predict(X_test_kpca)

#         accuracy = accuracy_score(y_test, y_pred)

#         precision_macro = precision_score(y_test, y_pred, average="macro", zero_division=0)
#         recall_macro = recall_score(y_test, y_pred, average="macro", zero_division=0)
#         f1_macro = f1_score(y_test, y_pred, average="macro", zero_division=0)

#         precision_weighted = precision_score(y_test, y_pred, average="weighted", zero_division=0)
#         recall_weighted = recall_score(y_test, y_pred, average="weighted", zero_division=0)
#         f1_weighted = f1_score(y_test, y_pred, average="weighted", zero_division=0)

#         update_best_model(
#             accuracy,
#             best_model,
#             scaler,
#             kpca,
#             "OAO",
#             kernel
#         )

#         all_results.append([
#             run+1,"OAO",kernel,
#             grid.best_score_,
#             accuracy,
#             precision_macro,
#             recall_macro,
#             f1_macro,
#             precision_weighted,
#             recall_weighted,
#             f1_weighted
#         ])

#         print(f"Accuracy: {accuracy:.4f}")


#     # =============================
#     # ALL TOGETHER
#     # =============================

#     print("\n▶ All-Together + Linear")

#     param_grid = {
#         "C":[1,10,100]
#     }

#     svm = LinearSVC(
#         multi_class="crammer_singer",
#         max_iter=2000
#     )

#     grid = GridSearchCV(
#         svm,
#         param_grid,
#         cv=5,
#         n_jobs=-1,
#         scoring="accuracy"
#     )

#     grid.fit(X_train_kpca, y_train)

#     best_model = grid.best_estimator_

#     y_pred = best_model.predict(X_test_kpca)

#     accuracy = accuracy_score(y_test, y_pred)

#     precision_macro = precision_score(y_test, y_pred, average="macro", zero_division=0)
#     recall_macro = recall_score(y_test, y_pred, average="macro", zero_division=0)
#     f1_macro = f1_score(y_test, y_pred, average="macro", zero_division=0)

#     precision_weighted = precision_score(y_test, y_pred, average="weighted", zero_division=0)
#     recall_weighted = recall_score(y_test, y_pred, average="weighted", zero_division=0)
#     f1_weighted = f1_score(y_test, y_pred, average="weighted", zero_division=0)

#     update_best_model(
#         accuracy,
#         best_model,
#         scaler,
#         kpca,
#         "All-Together",
#         "linear"
#     )

#     all_results.append([
#         run+1,"All-Together","linear",
#         grid.best_score_,
#         accuracy,
#         precision_macro,
#         recall_macro,
#         f1_macro,
#         precision_weighted,
#         recall_weighted,
#         f1_weighted
#     ])

#     print(f"Accuracy: {accuracy:.4f}")


# # =============================
# # RESULT TABLE
# # =============================

# df = pd.DataFrame(
#     all_results,
#     columns=[
#         "Run","Method","Kernel",
#         "CV Accuracy",
#         "Test Accuracy",
#         "Precision macro",
#         "Recall macro",
#         "F1 macro",
#         "Precision weighted",
#         "Recall weighted",
#         "F1 weighted"
#     ]
# )

# print("\nAverage Results")
# print(df.groupby(["Method","Kernel"]).mean()*100)


# # =============================
# # SAVE BEST MODEL
# # =============================

# print("\nSaving best model...")

# joblib.dump(best_global_model,"models/best_svm_model.pkl")
# joblib.dump(best_scaler,"models/scaler.pkl")
# joblib.dump(best_kpca,"models/kpca.pkl")

# print("Best Accuracy:",best_global_accuracy*100)
# print("Best Method:",best_method)
# print("Best Kernel:",best_kernel)

# print("Model saved in models/")


RUN 1/3

▶ OAA + rbf
Accuracy: 0.9722

▶ OAA + poly
Accuracy: 0.9444

▶ OAA + linear
Accuracy: 0.9722

▶ OAO + rbf
Accuracy: 0.9722

▶ OAO + poly
Accuracy: 0.9444

▶ OAO + linear
Accuracy: 0.9722

▶ All-Together + Linear
Accuracy: 1.0000

RUN 2/3

▶ OAA + rbf
Accuracy: 1.0000

▶ OAA + poly
Accuracy: 0.9444

▶ OAA + linear
Accuracy: 1.0000

▶ OAO + rbf
Accuracy: 1.0000

▶ OAO + poly
Accuracy: 0.9444

▶ OAO + linear
Accuracy: 1.0000

▶ All-Together + Linear
Accuracy: 1.0000

RUN 3/3

▶ OAA + rbf
Accuracy: 0.9444

▶ OAA + poly
Accuracy: 0.9167

▶ OAA + linear
Accuracy: 1.0000

▶ OAO + rbf
Accuracy: 0.9444

▶ OAO + poly
Accuracy: 0.9167

▶ OAO + linear
Accuracy: 1.0000

▶ All-Together + Linear
Accuracy: 1.0000

Average Results
                       Run  CV Accuracy  Test Accuracy  Precision macro  \
Method       Kernel                                                       
All-Together linear  200.0    92.500000     100.000000       100.000000   
OAA          linear  200.0    90.714286  